# HR Analytics AI System
## Notebook 2: Data Preprocessing

In this notebook, we will prepare the dataset for machine learning by:

1. Loading the cleaned Parquet data from the previous step
2. Identifying and handling missing values
3. Detecting and removing duplicate records
4. Dropping irrelevant columns that add no predictive value
5. Encoding categorical variables into numerical format

Clean data is the foundation of any reliable machine learning model.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

Libraries imported successfully.


### 1. Initialize Spark Session

We reuse the same Spark Session configuration from the previous notebook
to ensure consistency across all stages of the pipeline.

In [2]:
spark = SparkSession.builder \
    .appName("HR Analytics - Data Preprocessing") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session Created Successfully.")
print(f"Spark Version: {spark.version}")

Spark Session Created Successfully.
Spark Version: 3.5.8


### 2. Load Data

We load the dataset from the Parquet file saved in the previous notebook.
Parquet preserves the schema and data types, so no additional configuration is needed.

In [3]:
df = spark.read.parquet(
    r"C:\HR-Analytics-AI-System\data\hr_data.parquet"
)

print("Dataset Loaded Successfully.")
print(f"Total Rows: {df.count()}")
print(f"Total Columns: {len(df.columns)}")

Dataset Loaded Successfully.
Total Rows: 1470
Total Columns: 35


### 3. Handling Missing Values

Missing values can negatively impact model performance.
We first check how many missing values exist in each column,
then decide how to handle them appropriately.

In [4]:
print("Missing Values per Column:")
print("-" * 40)
missing = df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in df.columns
])
missing.show(vertical=True)

Missing Values per Column:
----------------------------------------
-RECORD 0-----------------------
 Age                      | 0   
 Attrition                | 0   
 BusinessTravel           | 0   
 DailyRate                | 0   
 Department               | 0   
 DistanceFromHome         | 0   
 Education                | 0   
 EducationField           | 0   
 EmployeeCount            | 0   
 EmployeeNumber           | 0   
 EnvironmentSatisfaction  | 0   
 Gender                   | 0   
 HourlyRate               | 0   
 JobInvolvement           | 0   
 JobLevel                 | 0   
 JobRole                  | 0   
 JobSatisfaction          | 0   
 MaritalStatus            | 0   
 MonthlyIncome            | 0   
 MonthlyRate              | 0   
 NumCompaniesWorked       | 0   
 Over18                   | 0   
 OverTime                 | 0   
 PercentSalaryHike        | 0   
 PerformanceRating        | 0   
 RelationshipSatisfaction | 0   
 StandardHours            | 0   
 StockOp

### 4. Handling Duplicate Records

Duplicate records can bias the model and lead to incorrect predictions.
We check for any duplicated rows and remove them if found.

In [5]:
total_rows = df.count()
unique_rows = df.distinct().count()
duplicates = total_rows - unique_rows

print(f"Total Rows:   {total_rows}")
print(f"Unique Rows:  {unique_rows}")
print(f"Duplicates:   {duplicates}")

if duplicates > 0:
    df = df.distinct()
    print(f"Duplicates removed. Remaining Rows: {df.count()}")
else:
    print("No duplicate records found. Dataset is clean.")

Total Rows:   1470
Unique Rows:  1470
Duplicates:   0
No duplicate records found. Dataset is clean.


### 5. Dropping Irrelevant Columns

Some columns provide no predictive value and should be removed
to reduce noise and improve model performance.

The following columns are dropped for the reasons below:

- **EmployeeCount:** All values are 1, no variation
- **StandardHours:** All values are 80, no variation
- **Over18:** All employees are over 18, no variation
- **EmployeeNumber:** Just an ID, not a predictive feature

In [6]:
cols_to_drop = ["EmployeeCount", "StandardHours", "Over18", "EmployeeNumber"]

df = df.drop(*cols_to_drop)

print("Irrelevant columns dropped successfully.")
print(f"Remaining Columns: {len(df.columns)}")
print("-" * 40)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

Irrelevant columns dropped successfully.
Remaining Columns: 31
----------------------------------------
 1. Age
 2. Attrition
 3. BusinessTravel
 4. DailyRate
 5. Department
 6. DistanceFromHome
 7. Education
 8. EducationField
 9. EnvironmentSatisfaction
10. Gender
11. HourlyRate
12. JobInvolvement
13. JobLevel
14. JobRole
15. JobSatisfaction
16. MaritalStatus
17. MonthlyIncome
18. MonthlyRate
19. NumCompaniesWorked
20. OverTime
21. PercentSalaryHike
22. PerformanceRating
23. RelationshipSatisfaction
24. StockOptionLevel
25. TotalWorkingYears
26. TrainingTimesLastYear
27. WorkLifeBalance
28. YearsAtCompany
29. YearsInCurrentRole
30. YearsSinceLastPromotion
31. YearsWithCurrManager


### 6. Encoding Categorical Variables

Machine learning models require numerical input.
Therefore, we need to convert categorical columns into numerical format.

The categorical columns in our dataset are:

- **Attrition:** Yes/No → 1/0
- **OverTime:** Yes/No → 1/0
- **Gender:** Male/Female → 1/0
- **BusinessTravel:** Non-Travel/Travel_Rarely/Travel_Frequently → 0/1/2
- **Department:** Research & Development/Sales/Human Resources → 0/1/2
- **EducationField:** Multiple categories → numerical labels
- **JobRole:** Multiple categories → numerical labels
- **MaritalStatus:** Single/Married/Divorced → 0/1/2

In [7]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

categorical_cols = [
    "Attrition", "OverTime", "Gender",
    "BusinessTravel", "Department",
    "EducationField", "JobRole", "MaritalStatus"
]

indexers = [
    StringIndexer(inputCol=col, outputCol=col + "_encoded", handleInvalid="keep")
    for col in categorical_cols
]

pipeline = Pipeline(stages=indexers)
df_encoded = pipeline.fit(df).transform(df)

print("Categorical columns encoded successfully.")
print("-" * 40)
for col in categorical_cols:
    print(f"{col} → {col}_encoded")

Categorical columns encoded successfully.
----------------------------------------
Attrition → Attrition_encoded
OverTime → OverTime_encoded
Gender → Gender_encoded
BusinessTravel → BusinessTravel_encoded
Department → Department_encoded
EducationField → EducationField_encoded
JobRole → JobRole_encoded
MaritalStatus → MaritalStatus_encoded


### Encoding Results

The StringIndexer assigns a numerical label to each unique category in a column.
The encoding is based on the frequency of each category, where the most frequent
category gets the label 0, the second most frequent gets 1, and so on.

For example:
- Attrition: No → 0, Yes → 1
- OverTime: No → 0, Yes → 1
- Gender: Male → 0, Female → 1

The original categorical columns are kept alongside the new encoded columns
so we can still reference the original values during analysis.

In [8]:
print("Sample of Encoded Columns:")
print("-" * 40)
df_encoded.select(
    "Attrition", "Attrition_encoded",
    "OverTime", "OverTime_encoded",
    "Gender", "Gender_encoded",
    "BusinessTravel", "BusinessTravel_encoded"
).show(5)

Sample of Encoded Columns:
----------------------------------------
+---------+-----------------+--------+----------------+------+--------------+-----------------+----------------------+
|Attrition|Attrition_encoded|OverTime|OverTime_encoded|Gender|Gender_encoded|   BusinessTravel|BusinessTravel_encoded|
+---------+-----------------+--------+----------------+------+--------------+-----------------+----------------------+
|      Yes|              1.0|     Yes|             1.0|Female|           1.0|    Travel_Rarely|                   0.0|
|       No|              0.0|      No|             0.0|  Male|           0.0|Travel_Frequently|                   1.0|
|      Yes|              1.0|     Yes|             1.0|  Male|           0.0|    Travel_Rarely|                   0.0|
|       No|              0.0|     Yes|             1.0|Female|           1.0|Travel_Frequently|                   1.0|
|       No|              0.0|      No|             0.0|  Male|           0.0|    Travel_Rarely|    

### Encoding Verification

The table above confirms that the encoding was applied correctly:

- Attrition: No → 0.0, Yes → 1.0
- OverTime: No → 0.0, Yes → 1.0
- Gender: Male → 0.0, Female → 1.0
- BusinessTravel: Travel_Rarely → 0.0, Travel_Frequently → 1.0

The encoded values are stored as doubles (0.0, 1.0, 2.0),
which is the standard format required by Spark ML models.

In [9]:
df_clean = df_encoded.drop(*categorical_cols)

print("Original categorical columns dropped successfully.")
print(f"Final Number of Columns: {len(df_clean.columns)}")
print("-" * 40)
for i, col in enumerate(df_clean.columns, 1):
    print(f"{i:2}. {col}")

Original categorical columns dropped successfully.
Final Number of Columns: 31
----------------------------------------
 1. Age
 2. DailyRate
 3. DistanceFromHome
 4. Education
 5. EnvironmentSatisfaction
 6. HourlyRate
 7. JobInvolvement
 8. JobLevel
 9. JobSatisfaction
10. MonthlyIncome
11. MonthlyRate
12. NumCompaniesWorked
13. PercentSalaryHike
14. PerformanceRating
15. RelationshipSatisfaction
16. StockOptionLevel
17. TotalWorkingYears
18. TrainingTimesLastYear
19. WorkLifeBalance
20. YearsAtCompany
21. YearsInCurrentRole
22. YearsSinceLastPromotion
23. YearsWithCurrManager
24. Attrition_encoded
25. OverTime_encoded
26. Gender_encoded
27. BusinessTravel_encoded
28. Department_encoded
29. EducationField_encoded
30. JobRole_encoded
31. MaritalStatus_encoded


### Dropping Original Categorical Columns

After encoding, the original categorical columns are no longer needed.
We drop them to keep the dataset clean and ready for machine learning.

The final dataset contains 31 columns:

- 23 original numerical columns
- 8 newly encoded categorical columns

All columns are now in numerical format, which is required by Spark ML models.

In [10]:
df_clean.write.mode("overwrite").parquet(
    r"C:\HR-Analytics-AI-System\data\hr_data_preprocessed.parquet"
)

print("Preprocessed dataset saved successfully.")
print(f"Location: C:\\HR-Analytics-AI-System\\data\\hr_data_preprocessed.parquet")
print(f"Total Rows: {df_clean.count()}")
print(f"Total Columns: {len(df_clean.columns)}")

Preprocessed dataset saved successfully.
Location: C:\HR-Analytics-AI-System\data\hr_data_preprocessed.parquet
Total Rows: 1470
Total Columns: 31


### 7. Summary

In this notebook, we successfully completed the data preprocessing phase:

1. Loaded the dataset from the Parquet file saved in the previous notebook
2. Checked for missing values — no missing values were found
3. Checked for duplicate records — no duplicates were found
4. Dropped 4 irrelevant columns that had no predictive value
5. Encoded 8 categorical columns into numerical format using StringIndexer
6. Dropped the original categorical columns after encoding
7. Saved the preprocessed dataset as a Parquet file for the next phase

The dataset is now fully cleaned and encoded, containing 1,470 rows and 31 columns.
It is ready for the next phase: Exploratory Data Analysis (EDA).